# Архитектура Cloudberry / Greenplum

Coordinator принимает SQL и строит распределённый план. Данные физически разделены между четырьмя primary-сегментами; mirrors и standby обеспечивают отказоустойчивость.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

Connecting to 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

## Топология

```text
Jupyter → coordinator cdw
  ├─ content 0,1 → sdw1; mirrors → sdw2
  └─ content 2,3 → sdw2; mirrors → sdw1
standby → scdw
PXF на sdw1/sdw2 → общий HDFS
```

`content=-1` обозначает coordinator; `p` — primary, `m` — mirror.

In [2]:
%%sql
SELECT dbid,content,role,preferred_role,mode,status,hostname,port
FROM gp_segment_configuration ORDER BY dbid;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

10 rows affected.

dbid,content,role,preferred_role,mode,status,hostname,port
1,-1,p,p,n,u,cdw,5432
2,0,p,p,s,u,sdw1,40000
3,1,p,p,s,u,sdw1,40001
4,2,p,p,s,u,sdw2,40000
5,3,p,p,s,u,sdw2,40001
6,0,m,m,s,u,sdw2,50000
7,1,m,m,s,u,sdw2,50001
8,2,m,m,s,u,sdw1,50000
9,3,m,m,s,u,sdw1,50001
10,-1,m,m,s,u,scdw,5432


## PostgreSQL и Greenplum: главное различие

PostgreSQL обычно выполняет запрос одним серверным процессом над одним экземпляром хранения. Greenplum сохраняет знакомый SQL и каталоги PostgreSQL, но распределяет пользовательские данные и вычисления между сегментами. Поэтому логически правильный SQL может иметь совершенно разную стоимость в зависимости от физического распределения.

В PostgreSQL основным вопросом часто становится «какой индекс использовать?». В Greenplum к нему добавляются вопросы:

- на каких сегментах находятся строки;
- требуется ли пересылка между сегментами;
- равномерно ли распределена работа;
- можно ли выполнить JOIN и агрегацию локально;
- не ограничивает ли весь запрос один перегруженный сегмент.

## MPP: shared-nothing

Greenplum — MPP-система (Massively Parallel Processing) с архитектурой shared-nothing. Каждый primary-сегмент имеет собственный процесс PostgreSQL, память и локальные файлы данных. Сегменты не читают файлы друг друга напрямую. Для совместной операции строки передаются через interconnect.

Преимущество — горизонтальный параллелизм: увеличение числа сегментов добавляет CPU, память и пропускную способность хранения. Цена — стоимость сетевого обмена и необходимость равномерно распределить данные.

## Coordinator

Coordinator принимает клиентское соединение, разбирает SQL, проверяет права, строит глобальный план и управляет выполнением. Он не является местом хранения обычных распределённых таблиц. На coordinator находятся глобальные системные каталоги и служебное состояние.

Coordinator не должен становиться вычислительным узким местом. Операции, которые собирают очень большой результат через `Gather Motion`, переносят работу и трафик к coordinator. Поэтому аналитический запрос обычно должен максимально фильтровать и агрегировать данные на сегментах.

## Primary, mirror и content

`content` — логический сегмент данных. У каждого content есть primary, который обслуживает запросы, и mirror — синхронная резервная копия. В стенде четыре content: 0–3.

`dbid` идентифицирует конкретный экземпляр процесса, а не логический набор данных. После failover бывший mirror может стать primary, но content останется тем же. `preferred_role` показывает исходно назначенную роль, `role` — текущую.

В `gp_segment_configuration`:

- `status='u'` — экземпляр доступен;
- `mode='s'` — primary и mirror синхронизированы;
- content `-1` — coordinator или standby coordinator.

## Standby coordinator

Standby хранит копию coordinator-каталогов и журнала, но не обслуживает обычные пользовательские запросы, пока не активирован. Он защищает от потери управляющего узла. Наличие standby не заменяет резервное копирование: логическая ошибка или удаление данных реплицируется вместе с корректными изменениями.

## Как проходит SELECT

1. Клиент отправляет SQL coordinator.
2. Parser и analyzer строят дерево запроса и разрешают имена объектов.
3. Optimizer выбирает порядок JOIN, методы доступа, агрегации и Motion.
4. План делится на slices в точках Motion.
5. Coordinator создаёт gang процессов исполнителей на сегментах.
6. Сегменты параллельно читают локальные строки и обмениваются данными.
7. Итоговый slice возвращает результат клиенту.

Время запроса определяется не средним сегментом, а самым медленным участником каждого зависимого этапа.

## Slices и gangs

Slice — часть плана между операторами Motion. Операторы одного slice могут выполняться конвейерно. Motion создаёт границу producer/consumer: один набор процессов отправляет строки, другой принимает.

Gang — группа segment processes, выделенная для slice. В плане номера slices помогают понять порядок распределённого выполнения. Большое число slices не обязательно плохо, но часто указывает на сложную цепочку обменов.

## Распределённый план

План делится на slices. `Gather Motion` собирает строки, `Redistribute Motion` пересылает их по хешу нового ключа, `Broadcast Motion` копирует небольшой набор каждому сегменту. Motion часто определяет стоимость JOIN и GROUP BY.

In [3]:
%%sql
EXPLAIN SELECT regioncountry,count(*)
FROM dds.ext_raw_yndx_metrica_logs GROUP BY regioncountry;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

9 rows affected.

QUERY PLAN
Gather Motion 4:1 (slice1; segments: 4) (cost=0.00..490.86 rows=1000 width=16)
-> Finalize HashAggregate (cost=0.00..490.81 rows=250 width=16)
Group Key: regioncountry
-> Redistribute Motion 4:4 (slice2; segments: 4) (cost=0.00..490.78 rows=250 width=16)
Hash Key: regioncountry
-> Streaming Partial HashAggregate (cost=0.00..490.77 rows=250 width=16)
Group Key: regioncountry
-> Foreign Scan on ext_raw_yndx_metrica_logs (cost=0.00..455.48 rows=250000 width=8)
Optimizer: Pivotal Optimizer (GPORCA)


## Interconnect

Interconnect — транспорт между segment processes. Через него проходят строки Motion. Даже если таблицы читаются быстро, большой Redistribute или Broadcast может ограничиваться сетью, сериализацией и очередями получателей.

Уменьшить трафик помогают ранний фильтр, частичная агрегация на сегментах, совместимое распределение больших таблиц и репликация только действительно маленьких справочников.

## Системные каталоги

Многие знакомые каталоги PostgreSQL остаются доступными: `pg_class`, `pg_attribute`, `pg_namespace`, `pg_stat_activity`. Greenplum добавляет:

- `gp_segment_configuration` — экземпляры кластера;
- `gp_distribution_policy` — физическая policy таблиц;
- `pg_exttable` — внешние таблицы;
- `gp_toolkit` — диагностические views и функции.

Не изменяйте системные каталоги напрямую. Физические свойства задаются DDL-командами.

In [4]:
%%sql
SELECT n.nspname AS schema_name,
       c.relname AS table_name,
       pg_get_table_distributedby(c.oid) AS distribution
FROM pg_class c
JOIN pg_namespace n ON n.oid=c.relnamespace
WHERE n.nspname='m_razhin' AND c.relkind='r'
ORDER BY c.relname
LIMIT 20;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

20 rows affected.

schema_name,table_name,distribution
m_razhin,countries,DISTRIBUTED REPLICATED
m_razhin,dm_rasp_largest_deals,DISTRIBUTED REPLICATED
m_razhin,yndx_metrica_logs_1_prt_1,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_10,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_11,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_12,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_13,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_14,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_15,DISTRIBUTED BY (visitid)
m_razhin,yndx_metrica_logs_1_prt_16,DISTRIBUTED BY (visitid)


## Транзакции и MVCC

Greenplum наследует MVCC PostgreSQL: изменение создаёт новую версию строки, а snapshot определяет видимость. Но транзакция распределена между coordinator и несколькими сегментами. Coordinator координирует согласованный commit.

Долгие транзакции мешают очистке старых версий на множестве сегментов. Массовые изменения следует проектировать как управляемые batch-операции и не оставлять соединение `idle in transaction`.

## Статистика и optimizer

Optimizer оценивает количество строк, селективность фильтров и стоимость Motion. Без актуальной статистики он может broadcast-ить большой набор, выбрать неудачный порядок JOIN или неверно оценить память. После значительной загрузки выполняют `ANALYZE` нужной таблицы.

`EXPLAIN` показывает оценку, `EXPLAIN ANALYZE` действительно выполняет запрос и показывает фактические строки/время. Изменяющий запрос с `EXPLAIN ANALYZE` также выполняет изменение — это важно помнить.

## Типы таблиц

В дальнейших модулях используются:

- heap — привычное построчное изменяемое хранение;
- append-optimized row — эффективная последовательная загрузка;
- append-optimized column — чтение выбранных колонок и сжатие;
- partitioned table — логическое дерево физических частей;
- external table — данные остаются во внешнем источнике, например HDFS или GPFDIST.

Способ хранения и distribution policy решают разные задачи и настраиваются совместно.

## PXF и GPFDIST

PXF — слой доступа сегментов к HDFS и другим внешним системам. Сегменты читают внешние фрагменты параллельно. GPFDIST — высокопроизводительный HTTP-сервер для параллельной загрузки/выгрузки файлов.

PXF не превращает HDFS-файл во внутреннюю таблицу: внешний запрос повторно обращается к источнику. Для часто используемых данных может быть выгодна загрузка во внутреннюю AO column table.

## Распределение и skew

Хеш ключа определяет сегмент. Хороший ключ имеет много значений, равномерную частоту и совпадает с частыми ключами JOIN. Самый загруженный сегмент ограничивает скорость всего запроса.

In [5]:
%%sql
SELECT gp_segment_id,count(*) rows_count
FROM m_razhin.yndx_metrica_logs GROUP BY gp_segment_id ORDER BY 1;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

4 rows affected.

gp_segment_id,rows_count
0,735843
1,733831
2,735895
3,735906


## Источники

- `dds.ext_raw_yndx_metrica_logs` — PXF/Метрика;
- `gpfdist://cdw:8080/countries.csv`;
- `/data/raw/m_razhin/` — персональный HDFS;
- `/moex_labs/raw/trades/` — MOEX;
- `m_razhin` — рабочая схема.

In [6]:
%%sql
SET search_path TO m_razhin,public;
SELECT current_database(),current_user,current_schema();
SELECT extname,extversion FROM pg_extension ORDER BY 1;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

1 rows affected.

4 rows affected.

extname,extversion
gp_exttable_fdw,1.0
gp_toolkit,1.4
plpgsql,1.0
pxf,2.0


## Самопроверка

Все 10 записей конфигурации должны иметь status `u`; PXF и обе учебные схемы должны существовать.

In [7]:
%%sql
SELECT count(*) FILTER(WHERE status='u') up,count(*) total FROM gp_segment_configuration;
SELECT EXISTS(SELECT 1 FROM pg_extension WHERE extname='pxf') pxf,
EXISTS(SELECT 1 FROM pg_namespace WHERE nspname='m_razhin') student_schema;

Running query in 'postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex'

1 rows affected.

1 rows affected.

pxf,student_schema
True,True
